## Silver Stock Prices Table

In [0]:
import dlt
from pyspark.sql.functions import (
    col, to_date, round as spark_round, when, lit, row_number
)
from pyspark.sql.window import Window

@dlt.table(
    name="stock_prices",
    comment="Cleaned and validated daily stock prices. Adjusted for splits, nulls handled, typed correctly.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_or_drop("valid_close_price", "close > 0")
@dlt.expect_or_drop("valid_volume", "volume >= 0")
@dlt.expect_or_drop("valid_symbol", "symbol IS NOT NULL AND symbol != ''")
@dlt.expect_or_drop("valid_date", "trade_date IS NOT NULL")
@dlt.expect("high_gte_low", "high >= low")
def stock_prices():
    """
    Reads from bronze.raw_ohlcv and produces a clean, deduplicated,
    properly typed Silver table.
    """
    # Read from the Bronze table
    bronze_ohlcv = spark.table("raghavan_trading_signals.bronze.raw_ohlcv")

    # Deduplicate: if the same stock+date appears multiple times
    # (from overlapping file drops), keep only the latest ingestion.
    dedup_window = Window.partitionBy("symbol", "date").orderBy(col("_ingested_at").desc())

    cleaned = (
        bronze_ohlcv
        # Cast and rename for clarity
        .withColumn("trade_date", to_date(col("date")))
        .withColumn("open", spark_round(col("open").cast("double"), 4))
        .withColumn("high", spark_round(col("high").cast("double"), 4))
        .withColumn("low", spark_round(col("low").cast("double"), 4))
        .withColumn("close", spark_round(col("close").cast("double"), 4))
        .withColumn("adj_close", spark_round(col("adj_close").cast("double"), 4))
        .withColumn("volume", col("volume").cast("long"))
        # Deduplicate
        .withColumn("_row_num", row_number().over(dedup_window))
        .filter(col("_row_num") == 1)
        .drop("_row_num")
        # Select final columns in a clear order
        .select(
            "symbol",
            "trade_date",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume",
            col("dividends").cast("double").alias("dividends"),
            col("stock_splits").cast("double").alias("stock_splits"),
            col("_ingested_at").alias("bronze_ingested_at"),
            col("_source_file").alias("bronze_source_file"),
        )
    )

    return cleaned

## Silver Macro Indicators Table

In [0]:
@dlt.table(
    name="macro_indicators",
    comment="Cleaned economic indicators from FRED. Forward-filled to trading days, typed correctly.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_or_drop("valid_value", "value IS NOT NULL")
@dlt.expect_or_drop("valid_indicator", "indicator IS NOT NULL")
@dlt.expect_or_drop("valid_date", "indicator_date IS NOT NULL")
def macro_indicators():
    """
    Reads from bronze.raw_macro and produces a clean, pivoted Silver table.
    Each row = one date, with columns for each economic indicator.
    """
    bronze_macro = spark.table("raghavan_trading_signals.bronze.raw_macro")

    # Deduplicate: keep latest ingestion per indicator+date
    dedup_window = Window.partitionBy("indicator", "date").orderBy(col("_ingested_at").desc())

    cleaned = (
        bronze_macro
        .withColumn("indicator_date", to_date(col("date")))
        .withColumn("value", col("value").cast("double"))
        .withColumn("_row_num", row_number().over(dedup_window))
        .filter(col("_row_num") == 1)
        .drop("_row_num")
        .select(
            "indicator",
            "series_id",
            "indicator_date",
            "value",
            col("_ingested_at").alias("bronze_ingested_at"),
        )
    )

    return cleaned

## Create a Pivoted Macro View

In [0]:
from pyspark.sql.functions import max as spark_max, last as spark_last

# Known macro indicators — kept static so Lakeflow can plan the DAG lazily.
# `GroupedData.pivot` is not supported inside declarative pipelines because
# it eagerly queries distinct values at plan time. We use conditional
# aggregation instead, which is fully lazy and equivalent.
MACRO_INDICATORS = [
    "vix",
    "fed_funds_rate",
    "treasury_10y",
    "treasury_2y",
    "usd_index",
    "cpi",
    "unemployment_rate",
]


@dlt.table(
    name="macro_daily_wide",
    comment="Wide macro data: one row per date, one column per indicator. Forward-filled.",
    table_properties={
        "quality": "silver"
    }
)
def macro_daily_wide():
    """
    Reshapes long-form macro data into wide format without using pivot.
    One row per trading date, one column per indicator, forward-filled.
    """
    macro = spark.table("raghavan_trading_signals.silver.macro_indicators")

    # Conditional aggregation: for each indicator, take its value when the
    # row matches that indicator name, else NULL, and aggregate per date.
    agg_exprs = [
        spark_max(
            when(col("indicator") == ind, col("value"))
        ).alias(ind)
        for ind in MACRO_INDICATORS
    ]

    wide = (
        macro
        .groupBy("indicator_date")
        .agg(*agg_exprs)
    )

    # Forward-fill each indicator column: economic indicators like CPI are
    # monthly. On days without a reading, carry forward the last known value.
    fill_window = Window.orderBy("indicator_date").rowsBetween(
        Window.unboundedPreceding, Window.currentRow
    )

    for ind in MACRO_INDICATORS:
        wide = wide.withColumn(
            ind,
            spark_last(col(ind), ignorenulls=True).over(fill_window)
        )

    return wide